# Decomposing SDT Criterion to DDM Sensory and Decisional Bias
**List of contents:**\
1- SJ\
2- TOJ

In [2]:
# Load required libraries
library(tidyverse)
library(ggpubr)
library(rstatix)
library(readxl)
library(dplyr)
library(broom)
library(DEoptim)
library(writexl)
library(MASS) 
library(WRS2)
library(boot)

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.3     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: 'rstatix'


The following object is masked from 'package:stats':

    filter


Warning message:
"package 'DEoptim' was built under R version 4.4.3"
Loading required package: parallel


DEoptim package
Differential Evolution algorithm in R
Authors: D. Ardia, K. Mullen, B. Peterson and J. Ulrich


Warning me

###############################################################\
###############################################################
# 1- SJ

In [3]:
df_observer_SJ <- read_xlsx("files/SJ_general_observer_parameters.xlsx")
head(df_observer_SJ)

participant,Clow,Chigh,sigma,PSS,delta_c
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,-246.1273,188.53599,57.15751,-28.79564,434.6633
3,-189.9685,54.52676,123.80209,-67.72085,244.4952
4,-254.6838,92.91935,174.46400,-80.88223,347.6032
5,-243.7031,217.30374,103.52649,-13.19967,461.0068
7,-269.4322,154.93041,107.24385,-57.25092,424.3626
8,-202.0173,109.56783,117.14745,-46.22473,311.5851


In [4]:
df_drift_SJ <- read_xlsx("files/SJ_sensory_bias_general.xlsx")
head(df_drift_SJ)

ID,positive_root,negative_root,max_snr_SOA,max_snr_value
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,193.2109,-227.5455,-46.33780,3.4371268
3,-103.9749,-103.9749,-103.97488,-0.1829786
4,123.2282,-310.6443,-199.98941,0.7380726
5,208.2550,-234.0175,-44.42087,1.4723633
7,168.7700,-259.7393,-52.88315,2.2317758
8,76.6718,-183.4941,-61.61357,0.7924913


In [5]:
df_bias_SJ <- read_xlsx("files/SJ_decision_bias_general.xlsx")
head(df_bias_SJ)

ID,mean_Z
<dbl>,<dbl>
1,0.21244569
3,0.42931124
4,-0.05707451
5,0.32700068
7,0.01437192
8,0.19693812


In [6]:
colnames(df_observer_SJ)[1] <- "ID"

In [7]:
final_df_SJ <- df_observer_SJ %>%
  left_join(df_drift_SJ, by = "ID") %>%
  left_join(df_bias_SJ, by = "ID")

In [8]:
final_df_SJ %>% identify_outliers(Chigh) %>% filter(is.extreme == TRUE)

ID,Clow,Chigh,sigma,PSS,delta_c,positive_root,negative_root,max_snr_SOA,max_snr_value,mean_Z,is.outlier,is.extreme
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>


In [9]:
final_df_SJ %>% identify_outliers(Clow) %>% filter(is.extreme == TRUE)

ID,Clow,Chigh,sigma,PSS,delta_c,positive_root,negative_root,max_snr_SOA,max_snr_value,mean_Z,is.outlier,is.extreme
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>


In [10]:
final_df_SJ %>% identify_outliers(positive_root) %>% filter(is.extreme == TRUE)

ID,Clow,Chigh,sigma,PSS,delta_c,positive_root,negative_root,max_snr_SOA,max_snr_value,mean_Z,is.outlier,is.extreme
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>


In [11]:
final_df_SJ %>% identify_outliers(negative_root) %>% filter(is.extreme == TRUE)

ID,Clow,Chigh,sigma,PSS,delta_c,positive_root,negative_root,max_snr_SOA,max_snr_value,mean_Z,is.outlier,is.extreme
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>


In [12]:
final_df_SJ %>% identify_outliers(mean_Z) %>% filter(is.extreme == TRUE)

ID,Clow,Chigh,sigma,PSS,delta_c,positive_root,negative_root,max_snr_SOA,max_snr_value,mean_Z,is.outlier,is.extreme
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>


In [13]:
model_Chigh <- lm(Chigh ~ positive_root + negative_root + mean_Z, data = final_df_SJ)
summary(model_Chigh)


Call:
lm(formula = Chigh ~ positive_root + negative_root + mean_Z, 
    data = final_df_SJ)

Residuals:
    Min      1Q  Median      3Q     Max 
-17.892 -11.665  -2.396  12.628  28.595 

Coefficients:
               Estimate Std. Error t value Pr(>|t|)    
(Intercept)    49.39856   22.03733   2.242 0.041710 *  
positive_root   0.73968    0.04914  15.052 4.86e-10 ***
negative_root   0.09280    0.07611   1.219 0.242921    
mean_Z        163.51734   37.89420   4.315 0.000712 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 15.9 on 14 degrees of freedom
Multiple R-squared:  0.9514,	Adjusted R-squared:  0.941 
F-statistic:  91.4 on 3 and 14 DF,  p-value: 1.962e-09


In [14]:
model_Clow <- lm(Clow ~ positive_root + negative_root  + mean_Z, data = final_df_SJ)
summary(model_Clow)


Call:
lm(formula = Clow ~ positive_root + negative_root + mean_Z, data = final_df_SJ)

Residuals:
    Min      1Q  Median      3Q     Max 
-36.297  -9.143  -0.831  13.043  37.719 

Coefficients:
                Estimate Std. Error t value Pr(>|t|)    
(Intercept)    -53.15087   28.25751  -1.881   0.0809 .  
positive_root    0.02297    0.06301   0.365   0.7209    
negative_root    0.72358    0.09760   7.414 3.28e-06 ***
mean_Z        -120.43895   48.59009  -2.479   0.0265 *  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 20.39 on 14 degrees of freedom
Multiple R-squared:  0.8248,	Adjusted R-squared:  0.7873 
F-statistic: 21.98 on 3 and 14 DF,  p-value: 1.462e-05


###############################################################\
###############################################################
# 2- TOJ

In [15]:
df_observer_TOJ <- read_xlsx('files/TOJ_general_observer_parameters.xlsx')
head(df_observer_TOJ)

participant,SD,criterion
<dbl>,<dbl>,<dbl>
1,240.7688,-52.585878
3,213.5215,-51.164074
5,189.5935,-101.185727
6,340.3783,-7.222642
7,221.0994,-90.866481
8,197.5471,-26.256731


In [16]:
df_drift_TOJ <- read_xlsx('files/TOJ_sensory_bias_general.xlsx')
head(df_drift_TOJ)

ID,root
<dbl>,<dbl>
1,-144.8011
3,25.7442
5,-123.8457
6,-26.1496
7,-26.6953
8,-24.2562


In [17]:
df_bias_TOJ <- read_xlsx("files/TOJ_decision_bias_general.xlsx")
head(df_bias_TOJ)

ID,mean_Z
<dbl>,<dbl>
1,-0.085608853
3,0.151250826
5,0.248222514
6,0.003489469
7,0.339033525
8,0.154883552


In [18]:
colnames(df_observer_TOJ)[1] <- "ID"

In [19]:
final_df_TOJ <- df_observer_TOJ %>%
  left_join(df_drift_TOJ, by = "ID") %>%
  left_join(df_bias_TOJ, by = "ID")
head(final_df_TOJ)

ID,SD,criterion,root,mean_Z
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,240.7688,-52.585878,-144.8011,-0.085608853
3,213.5215,-51.164074,25.7442,0.151250826
5,189.5935,-101.185727,-123.8457,0.248222514
6,340.3783,-7.222642,-26.1496,0.003489469
7,221.0994,-90.866481,-26.6953,0.339033525
8,197.5471,-26.256731,-24.2562,0.154883552


In [20]:
final_df_TOJ %>% identify_outliers(criterion) %>% filter(is.extreme == TRUE)

ID,SD,criterion,root,mean_Z,is.outlier,is.extreme
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>


In [21]:
final_df_TOJ %>% identify_outliers(root) %>% filter(is.extreme == TRUE)

ID,SD,criterion,root,mean_Z,is.outlier,is.extreme
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>


In [22]:
final_df_TOJ %>% identify_outliers(mean_Z) %>% filter(is.extreme == TRUE)

ID,SD,criterion,root,mean_Z,is.outlier,is.extreme
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>


In [23]:
model_criterion <- lm(criterion ~ root + mean_Z, data = final_df_TOJ)
summary(model_criterion)


Call:
lm(formula = criterion ~ root + mean_Z, data = final_df_TOJ)

Residuals:
    Min      1Q  Median      3Q     Max 
-43.016 -12.735   4.666  10.123  37.086 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept)    2.17701    6.50160   0.335    0.742    
root           0.55181    0.06356   8.681 3.10e-07 ***
mean_Z      -246.63437   38.94447  -6.333 1.34e-05 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 22.13 on 15 degrees of freedom
Multiple R-squared:  0.9256,	Adjusted R-squared:  0.9157 
F-statistic: 93.31 on 2 and 15 DF,  p-value: 3.441e-09
